**Part 1**

In [ ]:
import numpy as np
import pandas as pd

ratings_df = pd.read_csv("ratings.csv")

R_df = ratings_df.pivot(index="userId", columns="movieId", values="rating")

R_df = R_df.fillna(0)

R = R_df.to_numpy()


# 4. Function to compute user cosine similarity matrix
def user_similarity_matrix(R):
  # Dot product between all pairs of user vectors
  dot_products = np.dot(R, R.T)

  # L2 norm for each user vector
  norms = np.linalg.norm(R, axis=1, keepdims=True)

  # Outer product of norms gives the denominator matrix
  norm_matrix = np.dot(norms, norms.T)

  # Avoid division by zero for users with no ratings
  norm_matrix[norm_matrix == 0] = 1e-9

  # Return the user cosine similarity matrix
  return dot_products / norm_matrix


similarity_matrix = user_similarity_matrix(R)

print("Rating matrix shape (Users x Items):", R.shape)
print("User similarity matrix shape:", similarity_matrix.shape)
print("Sample of similarity matrix (first 5x5 users):\n", similarity_matrix[:5, :5])

Rating matrix shape (Users x Items): (610, 9724)
User similarity matrix shape: (610, 610)
Sample of similarity matrix (first 5x5 users):
 [[1.         0.02728287 0.05972026 0.19439477 0.12907989]
 [0.02728287 1.         0.         0.00372587 0.01661444]
 [0.05972026 0.         1.         0.00225139 0.00501972]
 [0.19439477 0.00372587 0.00225139 1.         0.1286589 ]
 [0.12907989 0.01661444 0.00501972 0.1286589  1.        ]]


Our dataset contains 610 users and 9,724 movies.

This results in a square matrix of size 610 by 610, representing the similarity score between every pair of users.

On the main diagonal, all values are 1.0, because every user is completely identical to themselves. The other numbers represent similarity scores between different users; higher values indicate more similar movie tastes.

**Part 2**


In [2]:
import numpy as np
import pandas as pd

ratings_df = pd.read_csv("ratings.csv")

R_df = ratings_df.pivot(index="userId", columns="movieId", values="rating")
R_df = R_df.fillna(0)
R = R_df.to_numpy()


def user_similarity_matrix(R):
  dot_products = np.dot(R, R.T)
  norms = np.linalg.norm(R, axis=1, keepdims=True)
  norm_matrix = np.dot(norms, norms.T)
  norm_matrix[norm_matrix == 0] = 1e-9
  return dot_products / norm_matrix


similarity_matrix = user_similarity_matrix(R)


def predict_rating(R, sim_matrix, user_idx, item_idx, k=2):
  # Find other users who have rated the item (rating > 0) excluding the target user
  other_users = np.where((R[:, item_idx] > 0) & (np.arange(len(R)) != user_idx))[
      0
  ]

  # If no other user has rated this item, return 0.0
  if len(other_users) == 0:
    return 0.0

  # Get similarities between the target user and candidate users
  similarities = sim_matrix[user_idx, other_users]

  # Select top-k most similar users (highest similarity scores)
  top_indices = np.argsort(similarities)[::-1][:k]
  top_users = other_users[top_indices]
  top_sims = similarities[top_indices]

  # Get the ratings given by these top-k users for the item
  top_ratings = R[top_users, item_idx]

  # Calculate the sum of absolute similarities for normalization
  sim_sum = np.sum(np.abs(top_sims))

  # Prevent division by zero if similarity sum is zero
  if sim_sum == 0:
    return 0.0

  return float(np.sum(top_sims * top_ratings) / sim_sum)


predicted_score = predict_rating(R, similarity_matrix, user_idx=0, item_idx=0, k=2)
print("Predicted Rating:", predicted_score)

Predicted Rating: 3.473577681146988


The output score of *3.47* out of *5* means that, based on the preferences of similar users who watched this movie, the system estimates the target user will likely enjoy it and give it a rating around *3.5*.

**Part 3**

In [ ]:
import numpy as np
import pandas as pd

ratings_df = pd.read_csv("ratings.csv")
R_df = ratings_df.pivot(index="userId", columns="movieId", values="rating")
R_df = R_df.fillna(0)
R = R_df.to_numpy()


# Function for Matrix Reconstruction with Truncated SVD
def svd_reconstruct(R, k):
  # Perform full SVD
  U, S, Vt = np.linalg.svd(R, full_matrices=False)

  U_k = U[:, :k]
  S_k = np.diag(S[:k])
  Vt_k = Vt[:k, :]

  R_hat = np.dot(np.dot(U_k, S_k), Vt_k)
  return R_hat


# Execute SVD reconstruction with k = 20 latent components
R_hat = svd_reconstruct(R, k=20)

print("Original Rating Matrix Shape:", R.shape)
print("Reconstructed Matrix Shape (R_hat):", R_hat.shape)
print("Sample of Reconstructed Matrix (first 5x5 ratings):\n", R_hat[:5, :5])

Original Rating Matrix Shape: (610, 9724)
Reconstructed Matrix Shape (R_hat): (610, 9724)
Sample of Reconstructed Matrix (first 5x5 ratings):
 [[ 2.29033555e+00  1.46020278e+00  1.03350661e+00 -6.13340663e-02
  -2.27547425e-03]
 [ 3.85700733e-02  1.52719248e-02  1.69677687e-02  2.94442743e-03
   1.92012443e-02]
 [-1.52196682e-02  4.90666929e-02  4.72015759e-02 -4.93618264e-03
  -3.53489111e-02]
 [ 2.23862066e+00  6.00106271e-02  3.93842306e-02  6.64554481e-02
   2.21805526e-01]
 [ 1.35836280e+00  9.70070602e-01  3.40938886e-01  1.21053007e-01
   4.79935808e-01]]


Both the original rating matrix and the reconstructed matrix *(R^)* have the exact same shape of 610 users by 9,724 movies. This proves that our SVD reconstruction successfully generated a complete matrix matching the dimensions of our dataset.

Unlike the original sparse matrix—which contained mostly zeros (unrated items) and whole numbers for ratings—the reconstructed matrix is dense. It replaces all the missing gaps with estimated, continuous decimal values (like 2.29 or 1.46). These numbers represent the system's prediction of how much a user would enjoy an unwatched movie.

You can see some small negative numbers in the sample (such as -6.13e-02, which is -0.06). Mathematically, these happen because of the linear algebra and matrix multiplication in SVD. In a recommendation system, a negative score indicates a strong mismatch or that the user would likely dislike that item.